# पाठ 09 - मेटा-संज्ञान डिज़ाइन पैटर्न


## सेटअप

यह नोटबुक Microsoft Agent Framework का उपयोग करके मेटाकॉग्निशन डिज़ाइन पैटर्न को दर्शाती है.

**पूर्वापेक्षाएँ:**
- Azure OpenAI तैनाती पर्यावरण चर के माध्यम से कॉन्फ़िगर की गई हो
- Azure CLI प्रमाणीकृत (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## मेटाकॉग्निशन क्या है?

मेटाकॉग्निशन **सोच के बारे में सोचना** है। AI एजेंट्स के संदर्भ में, इसका अर्थ ऐसे एजेंट बनाना है जो कर सकें:

- **आत्म-निरीक्षण** अपने ही आउटपुट और तर्क प्रक्रिया पर करना
- **त्रुटियों का पता लगाना** और चुपचाप विफल होने के बजाय सुचारू रूप से ठीक होना
- **मूल्यांकन करना** कि क्या उनके उत्तर पूर्ण और सहायक हैं
- **अनुकूलन करना** उनकी रणनीति जब एक प्रारंभिक दृष्टिकोण काम नहीं करता (उदा., बैकअप सिस्टम पर वापस जाना)

एक मेटाकॉग्निटिव एजेंट सिर्फ सवालों का जवाब नहीं देता — यह अपने प्रदर्शन की निगरानी करता है और तुरंत समायोजित कर लेता है।


## प्राथमिक और बैकअप टूल

एक सामान्य मेटाकॉग्निशन पैटर्न है **फॉलबैक रणनीति**। एजेंट पहले एक प्राथमिक टूल आज़माता है; यदि यह विफल हो जाता है (उदा., 404 त्रुटि), तो एजेंट विफलता को पहचानकर पारदर्शी रूप से एक बैकअप टूल पर स्विच कर देता है।

यह वास्तविक दुनिया की प्रणालियों का प्रतिबिंब है जहाँ प्राथमिक सेवाएँ अनुपलब्ध हो सकती हैं और एजेंटों को वैकल्पिक मार्ग चुनने से पहले समस्या का स्वयं निदान करना होता है।

नीचे हम दो उड़ान लुकअप टूल परिभाषित करते हैं:
- **प्राथमिक** — Paris, Tokyo, और Barcelona कवर करता है
- **बैकअप** — Berlin, Sydney, और New York City कवर करता है


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## त्रुटि पुनर्प्राप्ति के साथ आत्म-प्रतिबिंबित एजेंट

नीचे दिए गए एजेंट को निर्देश दिया गया है कि वह पहले प्राथमिक उड़ान प्रणाली का प्रयास करे, विफलताओं को पहचाने, और पारदर्शी रूप से बैकअप प्रणाली पर लौट जाए। प्रत्येक प्रतिक्रिया के बाद यह संक्षेप में स्वयं का मूल्यांकन करता है कि क्या उसने उपयोगकर्ता के प्रश्न का पूरा उत्तर दिया।


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## स्व-मूल्यांकन पैटर्न

मेटाकॉग्निशन का एक और पहलू **स्व-मूल्यांकन** है: एक अलग एजेंट (या उसी एजेंट द्वारा दूसरी बार) किसी उत्तर की पूर्णता, सटीकता, और सहायकता के लिए समीक्षा करता है।

नीचे हम एक `ResponseEvaluator` एजेंट बनाते हैं जो यात्रा-एजेंट के उत्तरों को तीन आयामों पर अंक देता है।


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## सारांश

इस पाठ में आपने Microsoft Agent Framework का उपयोग करके **मेटाकॉग्निटिव एजेंट** कैसे बनाते हैं, यह सीखा:

- **स्व-निरीक्षण**: ऐसे एजेंट जो अपनी ही तर्क प्रक्रिया की निगरानी करते हैं और पारदर्शी ढंग से बताते हैं कि क्या हुआ।
- **फॉलबैक के साथ त्रुटि पुनर्प्राप्ति**: एक प्राथमिक + बैकअप टूल पैटर्न जहाँ एजेंट असफलताओं का पता लगाता है (उदा., 404 त्रुटियाँ) और स्वचालित रूप से वैकल्पिक स्रोत आज़माता है।
- **आत्म-मूल्यांकन**: एक अलग मूल्यांकन एजेंट जो प्रतिक्रियाओं को पूर्णता, सटीकता, और सहायकता के लिए स्कोर करता है।

ये पैटर्न एजेंटों को अधिक मजबूत, पारदर्शी, और भरोसेमंद बनाते हैं — उत्पादन परिनियोजन के लिए महत्वपूर्ण गुण।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
यह दस्तावेज़ AI अनुवाद सेवा Co-op Translator (https://github.com/Azure/co-op-translator) का उपयोग करके अनुवादित किया गया है। हम सटीकता के लिए प्रयास करते हैं, परंतु कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में आधिकारिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए पेशेवर मानव अनुवाद की सलाह दी जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
